<!--
SPDX-FileCopyrightText: Copyright (c) 2026, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0
-->

# Varying an agent with Fabric

The [quickstart](01_quickstart.ipynb) ran one agent on one harness. Fabric's
value is that, once an agent is a typed config, you can change how it runs
without touching your application code. Those changes fall into two groups:

1. **Harness variation** — run the *same* logical agent on a different
   harness (Hermes, Codex, Claude, or Deep Agents). This is Fabric's primary
   purpose.
2. **Agent configuration variation** — vary fields of the `FabricConfig`
   itself: skills, MCP servers, models, telemetry, and so on.

A note on terms, used consistently below:

- A **harness** is the agent framework that runs the agent — Hermes, Codex,
  Claude, Deep Agents.
- An **adapter** is the Fabric integration that connects Fabric to a harness.
- The **runtime** is Fabric's own start/invoke/stop execution lifecycle.
  Switching harnesses does not change the runtime contract.

Where a harness's prerequisites are present, the cells below run it; where
they are not, they inspect the resolved config with `plan()` and say what to
provide — so the notebook always executes top to bottom.

<div align="center">
<img src="img/variations.svg" width="620" alt="Two ways to vary an agent with Fabric: harness variation and agent configuration variation.">
<br><br>
<em>Two kinds of variation on one typed agent — run it on a different harness, or vary <code>FabricConfig</code> fields like skills, MCP servers, models, and telemetry.</em>
</div>

## Setup

Load the environment. Each harness adapter runs in whatever Python
environment has that adapter installed; `settings["python"]` selects it. The
Hermes adapter lives in the repo's Hermes venv, and the others live in this
notebook's interpreter.

In [ ]:
import importlib.util
import os
import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    root = start.resolve()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent
    return root


def load_dotenv(path: Path) -> None:
    if not path.exists():
        return
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.removeprefix("export ").partition("=")
        key = key.strip()
        if key and key not in os.environ:
            os.environ[key] = value.strip().strip("'\"")


REPO_ROOT = find_repo_root(Path.cwd())
load_dotenv(REPO_ROOT / ".env")

from nemo_fabric import (
    EnvironmentConfig,
    Fabric,
    FabricConfig,
    HarnessConfig,
    MetadataConfig,
    ModelConfig,
    RelayAtofConfig,
    RelayObservabilityConfig,
    RuntimeConfig,
)

# A harness adapter runs in whatever Python environment has that adapter
# installed. The Hermes adapter lives in the repo's separate Hermes venv; the
# Codex, Claude, and Deep Agents adapters live in this notebook's interpreter.
FABRIC_PY = sys.executable
_hermes_python = REPO_ROOT / ".tmp" / "hermes-venv" / "bin" / "python"
HERMES_PY = str(_hermes_python) if _hermes_python.exists() else os.environ.get("ADAPTER_PYTHON")

# One shared workspace and one shared reviewer instruction, held constant below.
WORKSPACE = str(REPO_ROOT / "examples" / "code_review_agent" / "repos" / "my-service")
ARTIFACTS = REPO_ROOT / "examples" / "notebooks" / "artifacts" / "variations"
INSTRUCTION = "You are a concise code reviewer. Point out correctness bugs and risks."

fabric = Fabric()
RELAY_AVAILABLE = importlib.util.find_spec("nemo_relay") is not None
print("fabric interpreter :", FABRIC_PY)
print("hermes interpreter :", HERMES_PY or "not found")
print("nemo_relay present :", RELAY_AVAILABLE)


## Group 1 — Harness variation

One agent, one prompt, four harnesses. To make this a **controlled**
comparison, the portable agent is held constant — same identity, same
workspace, same output contract, same reviewer instruction — and only the
harness changes.

A few differences are unavoidable and are called out explicitly:

- **Model** — each harness talks to its own provider, so the model differs
  (Hermes and Deep Agents use an NVIDIA-hosted model; Codex uses OpenAI;
  Claude uses Anthropic). Fabric normalizes everything *around* the model.
- **Input schema** — the Codex adapter takes `text`; the others take `chat`.
- **Instruction delivery** — Hermes, Deep Agents, and Claude accept the
  system prompt via `harness.settings`; Codex takes its instruction from the
  input, so the prompt below carries the task for every harness.

`make_agent()` builds the shared config; the per-harness table supplies only
the values that must differ.

In [ ]:
def make_agent(harness) -> FabricConfig:
    """Build the SAME code-review agent for one harness.

    Everything in the body is portable and identical for every harness. Only the
    per-harness values (adapter, model, input schema, and the settings that
    harness genuinely requires) come from `harness`.
    """
    return FabricConfig(
        # --- portable: identical for every harness ---
        metadata=MetadataConfig(name="code-reviewer", description="Reviews code for correctness."),
        environment=EnvironmentConfig(provider="local", workspace=WORKSPACE, artifacts=str(ARTIFACTS)),
        runtime=RuntimeConfig(input_schema=harness["input_schema"], output_schema="message", artifacts=str(ARTIFACTS)),
        # --- harness-specific: necessarily differs (see notes above) ---
        harness=HarnessConfig(
            adapter_id=harness["adapter_id"],
            resolution="preinstalled",
            settings={**harness["settings"], "python": harness["python"]},
        ),
        models={"default": harness["model"]},
    )


NVIDIA_MODEL = ModelConfig(
    provider="nvidia", model="nvidia/nemotron-3-nano-30b-a3b",
    temperature=0.0, api_key_env="NVIDIA_API_KEY",
)

HARNESSES = [
    {"name": "Hermes", "adapter_id": "nvidia.fabric.hermes.sdk", "python": HERMES_PY,
     "input_schema": "chat", "model": NVIDIA_MODEL, "key": "NVIDIA_API_KEY",
     "needs": "a Hermes install (repo README's Hermes quick start) + NVIDIA_API_KEY",
     "settings": {"system_prompt": INSTRUCTION, "workspace": WORKSPACE,
                  "base_url": "https://integrate.api.nvidia.com/v1",
                  "max_iterations": 1, "max_tokens": 512,
                  "reasoning_config": {"effort": "none"}, "enabled_toolsets": []}},
    {"name": "Deep Agents", "adapter_id": "nvidia.fabric.langchain.deepagents", "python": FABRIC_PY,
     "input_schema": "chat", "model": NVIDIA_MODEL, "key": "NVIDIA_API_KEY",
     "needs": "the Deep Agents adapter (nemo-fabric[deepagents]) + NVIDIA_API_KEY",
     "settings": {"system_prompt": INSTRUCTION, "workspace": WORKSPACE}},
    {"name": "Codex", "adapter_id": "nvidia.fabric.codex.cli", "python": FABRIC_PY,
     "input_schema": "text", "key": "OPENAI_API_KEY", "binary": "codex",
     "model": ModelConfig(provider="openai", model="openai/gpt-5.4"),
     "needs": "an authenticated Codex CLI (`codex`) + OPENAI_API_KEY",
     "settings": {"sandbox": "workspace-write", "skip_git_repo_check": True}},
    {"name": "Claude", "adapter_id": "nvidia.fabric.claude", "python": FABRIC_PY,
     "input_schema": "chat", "key": "ANTHROPIC_API_KEY",
     "model": ModelConfig(provider="anthropic", model="anthropic/claude-sonnet-4-5",
                          api_key_env="ANTHROPIC_API_KEY"),
     "needs": "the Claude adapter + ANTHROPIC_API_KEY",
     "settings": {"system_prompt": INSTRUCTION, "permission_mode": "dontAsk"}},
]


def blocker(harness):
    """Return why a harness cannot run here, or None if it can."""
    py = harness.get("python")
    if not py or not os.path.exists(py):
        return "adapter interpreter not found"
    if harness.get("key") and not os.environ.get(harness["key"]):
        return f"{harness['key']} is not set"
    if harness.get("binary") and not shutil.which(harness["binary"]):
        return f"`{harness['binary']}` not on PATH"
    return None


def oneline(text, limit=200):
    s = " ".join(str(text).split())
    return s if len(s) <= limit else s[:limit] + " ..."


In [ ]:
PROMPT = "In one sentence, what does the Python expression sum(v) / len(v) compute, and name one risk?"

for harness in HARNESSES:
    cfg = make_agent(harness)
    plan = fabric.plan(cfg)
    rc = plan.effective_config.config.runtime
    model = plan.effective_config.config.models["default"]["model"]
    print(f"### {harness['name']}")
    print(f"    adapter = {plan.adapter.adapter_id}")
    print(f"    model   = {model}   input schema = {rc.input_schema}")
    reason = blocker(harness)
    if reason:
        print(f"    NOT RUN here ({reason}).")
        print(f"    To run it, provide: {harness['needs']}")
    else:
        try:
            result = await fabric.run(cfg, input=PROMPT)
            print(f"    RAN -> {result.status}")
            print(f"    reply: {oneline(getattr(result.output, 'response', result.output))}")
        except Exception as error:
            print(f"    run failed ({type(error).__name__}: {oneline(error, 120)}).")
            print(f"    To run it, provide: {harness['needs']}")
    print()

## Group 2 — Agent configuration variation

The other kind of variation keeps the harness fixed and changes fields of the
`FabricConfig`. Skills, MCP servers, models, and telemetry are all just
config — you compose them onto a **copy** of the agent, and the original is
untouched. Here we start from the Hermes agent and vary several fields at
once.

In [ ]:
SKILL = str(REPO_ROOT / "examples" / "code_review_agent" / "skills" / "code-review")

def summarize(cfg):
    return {
        "model": cfg.models["default"].model,
        "skills": [Path(p).name for p in (cfg.skills.paths if cfg.skills else [])],
        "mcp_servers": list(cfg.mcp.servers) if cfg.mcp else [],
    }

base = make_agent(HARNESSES[0])  # the Hermes agent
print("base      :", summarize(base))

# Vary fields on a deep copy — add a skill and an MCP server, swap the model.
variant = base.model_copy(deep=True)
variant.add_skill_path(SKILL)
variant.add_mcp_server(
    "docs", transport="streamable-http", url="${DOCS_MCP_URL}", exposure="fabric_managed"
)
variant.models["default"] = ModelConfig(
    provider="nvidia", model="nvidia/nemotron-3-super-49b-a5b", api_key_env="NVIDIA_API_KEY"
)
print("varied    :", summarize(variant))

print("base intact:", summarize(base))

### Telemetry is a configuration variation too

Turning on NeMo Relay tracing is the same pattern: clone the agent and set a
field. `enable_relay(...)` is a public `FabricConfig` method — no example
helper involved — so you can see exactly what telemetry adds to the config.
When the Relay dependency is present in the adapter environment, the run
writes trace files, which we then print in full.

We use the Deep Agents agent here because its adapter environment has
`nemo_relay` installed.

In [ ]:
import json

relay_dir = REPO_ROOT / "examples" / "notebooks" / "artifacts" / "relay"
deep = HARNESSES[1]  # Deep Agents
reason = blocker(deep)

if reason or not RELAY_AVAILABLE:
    missing = reason or "nemo_relay is not installed in the adapter environment"
    print(f"Not emitting telemetry here ({missing}).")
    print("Install the Relay extra (nemo-fabric[relay]) in the adapter env to")
    print("produce ATOF trace files under", relay_dir)
else:
    if relay_dir.exists():
        shutil.rmtree(relay_dir)
    # Telemetry is just another field: clone the agent and enable Relay.
    traced = make_agent(deep).model_copy(deep=True)
    traced.enable_relay(
        output_dir=str(relay_dir),
        observability=RelayObservabilityConfig(
            atof=RelayAtofConfig(
                enabled=True, output_directory=str(relay_dir),
                filename="events.atof.jsonl", mode="overwrite",
            ),
        ),
    )
    try:
        result = await fabric.run(traced, input=PROMPT)
    except Exception as error:
        print(f"Telemetry run failed ({type(error).__name__}: {oneline(error, 120)}).")
        print("To emit traces, provide:", deep["needs"])
    else:
        print("run status     :", result.status)
        print("telemetry refs :", [t.provider for t in result.telemetry] or "(none)")
        traces = sorted(p for p in relay_dir.rglob("*") if p.is_file())
        print("trace files    :", len(traces))
        for path in traces:
            print("   ", path.relative_to(REPO_ROOT))

        # Print the full Relay trace: every ATOF event.
        for path in traces:
            print(f"\n===== {path.relative_to(REPO_ROOT)} =====")
            lines = [line for line in path.read_text().splitlines() if line.strip()]
            for i, line in enumerate(lines, 1):
                print(f"\n--- event {i}/{len(lines)} ---")
                print(json.dumps(json.loads(line), indent=2))

## Recap

Two kinds of variation, one typed config:

- **Harness variation** — the same agent ran on different harnesses and
  returned the same `RunResult` shape, so your application code does not
  change when you compare or switch harnesses.
- **Agent configuration variation** — skills, MCP servers, models, and
  telemetry are all fields you compose onto a copy of the config, which is
  the basis for evaluation and ablation sweeps.

Throughout, Fabric's runtime contract (start, invoke, stop, and the
normalized result) stayed the same — only the harness or the configuration
changed. For the full API, see the [Python SDK guide](../../docs/sdk/python.mdx).